PENDING: connect two ramzis through a straight waveguide

In [ ]:
import gdsfactory as gf
import numpy as np
import cspdk.si220.cband
from axiomatic.pic_helpers import plot_circuit
cspdk.si220.cband.activate_pdk()

pdk = gf.get_active_pdk()
def ring_resonator(coupling_length: float, target_fsr: float, center_wvl: float, bend_radius: float = 5) -> gf.Component:
    n_eff = 2.38
    n_g = 4.3

    cavity_length_target_fsr = center_wvl**2 / (n_g * target_fsr * 1e-3)

    m = round(n_eff * cavity_length_target_fsr / center_wvl)
    ring_length_target = m * center_wvl / n_eff  # Actual round-trip length for resonance at 1.55 um
    length_arc = 2 * np.pi * bend_radius

    c = gf.Component()

    # Adjust length_y to maintain constant round-trip length
    length_x = coupling_length
    length_y = (ring_length_target - length_arc - 2 * length_x)/2

    ring = c << pdk.get_component(
        'ring_single',
        radius=bend_radius,
        gap=0.2,
        length_x=length_x,
        length_y=length_y
    )
    ring.name = "ring"

    ring.rotate(0).move((0.0, 0.0))

    # Add ports
    c.add_port("in0", port=ring.ports["o1"])
    c.add_port("out0", port=ring.ports["o2"])

    return c
    

def ramzi_filter(
    coupling_lengths: list[float],
    target_fsrs: list[float],
    coupling_gap: float = 0.2,
    center_wavelength: float = 1.55,
    n_g: float = 4.3,
    n_eff: float = 2.38,
    bend_radius: float = 5.0,
    phase_shifter_length: float = 10.0,  # µm, placeholder space for PS,
    include_gratings: bool = False,
    stage_spacing: float = 100.0  # µm, spacing between stages
) -> gf.Component:
    """
    Passive RAMZI filter implementation with CSPDK components.
    
    Args:
        coupling_lengths: List of coupler lengths (µm), one per ring.
        target_fsrs: List of FSRs (nm), one per ring.
        coupling_gap: Coupler gap (µm).
        center_wavelength: Center wavelength (µm).
        n_g: Group index.
        n_eff: Effective index.
        bend_radius: Bend radius (µm).
        phase_shifter_length: Length of straight waveguide placeholder for PS (µm).
    
    Returns:
        gf.Component: RAMZI filter layout.
    """
    assert len(coupling_lengths) == len(target_fsrs), \
        "Each ring must have a coupling length and target FSR"

    N = len(coupling_lengths)  # Number of rings
    c = gf.Component()

    input_mzi = c << pdk.get_component("mzi", delta_length=0)
    output_mzi = c << pdk.get_component("mzi", delta_length=0)

    # Keep track of ports to connect
    upper_port = input_mzi.ports["o3"]
    lower_port = input_mzi.ports["o4"]

    # Cascade of rings
    for i in range(N):
        # Calculate loop length from FSR
        target_fsr_um = target_fsrs[i] * 1e-3
        upper_ring = c << ring_resonator(
            coupling_length=coupling_lengths[i],
            target_fsr=target_fsr_um,
            center_wvl=center_wavelength,
            bend_radius=bend_radius,
        )
        lower_ring = c << ring_resonator(
            coupling_length=coupling_lengths[i],
            target_fsr=target_fsr_um,
            center_wvl=center_wavelength,
            bend_radius=bend_radius,
        )

        upper_ring.move(((upper_port.x + stage_spacing), bend_radius-5))
        lower_ring.mirror_y()
        lower_ring.move(((lower_port.x + stage_spacing), -bend_radius+5))

        gf.routing.route_single(
            c,
            upper_port,
            upper_ring.ports["in0"],
            cross_section=pdk.get_cross_section("strip"),
        )
        gf.routing.route_single(
            c,
            lower_port,
            lower_ring.ports["in0"],
            cross_section=pdk.get_cross_section("strip"),
        )

        # Update ports for next iteration
        upper_port = upper_ring.ports["out0"]
        lower_port = lower_ring.ports["out0"]

    # # Output MZI
    output_mzi.move((upper_port.x + stage_spacing, 0))
    waypoints_upper = [(output_mzi.ports["o2"].x - bend_radius, upper_port.y), (output_mzi.ports["o2"].x - bend_radius, output_mzi.ports["o2"].y)]
    gf.routing.route_single(
        c,
        upper_port,
        output_mzi.ports["o2"],
        waypoints=waypoints_upper,
        cross_section=pdk.get_cross_section("strip"),
    )
    waypoints_lower = [(output_mzi.ports["o1"].x - bend_radius, lower_port.y), (output_mzi.ports["o1"].x - bend_radius, output_mzi.ports["o1"].y)]
    gf.routing.route_single(
        c,
        lower_port,
        output_mzi.ports["o1"],
        waypoints= waypoints_lower,
        cross_section=pdk.get_cross_section("strip"),
    )

    # Add input/output ports
    c.add_port("in0", port=input_mzi.ports["o1"])
    c.add_port("in1", port=input_mzi.ports["o2"])
    c.add_port("out0", port=output_mzi.ports["o4"])
    c.add_port("out1", port=output_mzi.ports["o3"])

    return c

# Example 3-ring RAMZI
ramzi = ramzi_filter(
    coupling_lengths=[10, 12, 10],
    target_fsrs=[3000, 3000, 3000],
    center_wavelength=1.55,
    bend_radius=20,
    stage_spacing=100.0,
    include_gratings=False
)
plot_circuit(ramzi)